In [216]:
cd Prosody2Vec/

[Errno 2] No such file or directory: 'Prosody2Vec/'
/home/dcor/niskhizov/Prosody2Vec


In [217]:
import torch

In [218]:
# import clearml
# clearml.browser_login()

In [219]:
# from clearml import Task
# task = Task.init(project_name="my project", task_name="my task")

In [220]:
from torch import nn
import torch 
import glob
from IPython.display import clear_output, display, Audio
import copy

In [221]:
import torch

In [222]:
import model

In [223]:
# acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_soft", trust_repo=True).cuda()
# acoustic = model.AcousticModel(discrete=True).cuda()

In [224]:
# data_dir = './Emotion Speech Dataset/'
# data_dir = '/home/dcor/niskhizov/Prosody2Vec/IEMOCAP_full_release/'
data_dir = '/home/dcor/niskhizov/Prosody2Vec/Emotion Speech Dataset/'
# scan recursively for all .wav files in the data_dir
wav_files = glob.glob(data_dir + '/**/*.wav', recursive=True)



In [225]:
len(wav_files)

35000

In [226]:
# embeddings_dir = 'esd_female_018'
embeddings_dir = './esd_3sec_embeddings'

In [227]:
# create pytorch dataset that loads pairs of wav a and embeddings from iemocap_embeddings
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import pickle
import torchaudio

class IemocapDataset(Dataset):
    def __init__(self, audio_files):
        self.audio_files = []
        self.embeddings_file = []

        for audio_file in audio_files:
            out_file = f"{embeddings_dir}/{audio_file.split('/')[-1].replace('.wav', '.pkl')}"
            if os.path.exists(out_file):                
                self.embeddings_file.append(out_file)
                self.audio_files.append(audio_file)

    def __len__(self):
        return len(self.embeddings_file)
    
    def __getitem__(self, idx):

        wav_path = self.audio_files[idx]

        out_file = self.embeddings_file[idx]

        with open(out_file, 'rb') as f:
            embd = pickle.load(f)

        wav,sr = torchaudio.load(wav_path)

        # take the first 3 seconds of the audio

        wav = wav[:, :3*sr]

        
        
        return wav, embd

In [308]:


class FusionDecoderV3(nn.Module):
    def __init__(self, hidden_dim, acoustic):
        super(FusionDecoderV3, self).__init__()
        # self.attn = AttentionFusion(prosody_dim, hidden_dim)
        
        self.ff1 = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 128))
        self.ff2 = nn.Sequential(nn.Linear(512+256, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 512))
        self.ff3 = nn.Sequential(nn.Linear(192, 128), nn.ReLU(), nn.Linear(128, 128))

        self.base_model = copy.deepcopy(acoustic)

    def forward(self, units, emo_vecs, spk_vecs, logmels):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
        # logmels: (batch_size, time, n_mels)
        
        # batch_size, time, _ = units.shape
        
        # Apply attention
        o = self.base_model.encoder(units.cuda())

        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        o2b = self.ff3(spk_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2, o2b], dim=-1))

        d = self.base_model.decoder(o3, logmels)

        return d
    
    def generate(self, units, emo_vecs, spk_vecs):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
                
        # Apply attention
        o = self.base_model.encoder(units.cuda())

        # units_attn = self.attn(o, emo_vecs)
        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        o2b = self.ff3(spk_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)

        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2, o2b], dim=-1))


        d = self.base_model.decoder.generate(o3)
        
        return d


In [302]:
import time

In [303]:
wav_files_english = [x for x in wav_files if int(x.split('/')[-3]) > 10] 

In [304]:
ds = IemocapDataset(wav_files_english)

In [305]:
train_ds, test_ds = torch.utils.data.random_split(ds, [int(0.8*len(ds)), len(ds) - int(0.8*len(ds))])

In [306]:
# create collate function that will pad the sequences to the same length
def collate_fn(batch):
    wavs = [item[0][0] for item in batch]
    
    d_units, units, emo_vecs, spk_vecs, logmels = [], [], [], [], []
    for item in batch:
        d = item[1]['discrite_units']
        u = item[1]['units']
        mel = item[1]['logmel'].T

        d_units.append(d)
        units.append(u)
        emo_vecs.append(torch.tensor((item[1]['emo_vec'])))
        spk_vecs.append(item[1]['spk_vec'])

        mel  = mel[:u.size(0)*2,:]
        # print(mel.shape)
        mel = torch.nn.functional.pad(mel, (0,0,1,0))
        # print(mel.shape)

        logmels.append(mel)

    
    mels_lengths = torch.tensor([x.size(0) - 1 for x in logmels])
    units_lengths = torch.tensor([x.size(0) for x in units])

    d_units_padded = nn.utils.rnn.pad_sequence(d_units, batch_first=True, padding_value=-1)
    units_padded = nn.utils.rnn.pad_sequence(units, batch_first=True)
    logmels_padded = nn.utils.rnn.pad_sequence(logmels, batch_first=True)
    
    _,T,_ = units_padded.shape
    # pad the sequences

    wavs = nn.utils.rnn.pad_sequence(wavs, batch_first=True)

    
    return wavs, d_units_padded, units_padded, torch.stack(emo_vecs), torch.stack(spk_vecs), logmels_padded, mels_lengths, units_lengths

In [234]:
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=10)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=10)

In [309]:
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_discrete", trust_repo=True).cuda()
decoder = FusionDecoderV3(512, acoustic).cuda()


Using cache found in /home/dcor/niskhizov/cache/hub/bshall_acoustic-model_main


In [310]:
from torch.optim import Adam
from torch.nn.functional import l1_loss

optimizer = Adam(decoder.parameters(), lr=1e-4)


In [311]:
from tqdm import tqdm_notebook,tqdm

In [ ]:
for epoch in range(0,300000):  
    decoder.train()
    for idx,batch in tqdm(enumerate(train_dl),total=len(train_dl)):
        wavs, d_units_padded, units_padded, emo_vecs, spk_vecs, logmels_padded, mels_lengths, units_lengths  = batch
        
        optimizer.zero_grad()

        out =  decoder(d_units_padded.cuda(),emo_vecs.cuda(),spk_vecs,logmels_padded[:, 1:, :].cuda())
        # out = decoder(d_units_padded.cuda(), emo_vecs.cuda(), spk_vecs.cuda(), logmels_padded[:, :-1, :].cuda())
        # out = acoustic(units_padded.cuda(), logmels_padded[:, :-1, :].cuda())

        # target = hifigan(out[:1,:,:].transpose(1, 2))
        loss = l1_loss(out, logmels_padded[:, 1:, :].cuda(), reduction="none")
        loss = torch.sum(loss, dim=(1, 2)) / (out.size(-1) * mels_lengths.cuda())
        loss = torch.mean(loss)
        loss.backward()

        optimizer.step()

    if epoch % 500 == 0:
        print('Epoch:', epoch, 'Batch:', idx)
        print('Loss:', loss.item())

    if epoch % 500 == 0:
        torch.save(decoder.state_dict(), f"decoder_simple2_with_speaker_{epoch}.pth")

100%|██████████| 69/69 [00:23<00:00,  2.97it/s]


Epoch: 0 Batch: 68
Loss: 0.5632192492485046


100%|██████████| 69/69 [00:21<00:00,  3.28it/s]

Epoch: 500 Batch: 68
Loss: 0.3017868101596832



100%|██████████| 69/69 [00:20<00:00,  3.32it/s]


...

Epoch: 2000 Batch: 68
Loss: 0.2438010722398758


100%|██████████| 69/69 [00:20<00:00,  3.29it/s]

Epoch: 2500 Batch: 68
Loss: 0.23573856055736542



 42%|████▏     | 29/69 [00:10<00:13,  2.86it/s]


KeyboardInterrupt: 

In [548]:
epoch

2916

In [313]:
import plotly.express as px


In [314]:
px.imshow(out[0].detach().cpu().numpy().T)

## Inference

In [ ]:
ls -lash --sort time | grep decoder

 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 23:01 decoder_20.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:50 decoder_10.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:39 decoder_0.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:18 decoder_260.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:16 decoder_250.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:13 decoder_240.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:11 decoder_230.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:08 decoder_220.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:05 decoder_210.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:02 decoder_200.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 22:00 decoder_190.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 21:57 decoder_180.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 21:54 decoder_170.pth
 80M -rw-r--r--  1 niskhizov cs_dcor  80M Feb 26 21:52 decoder_160.pth
 80M -rw-r

In [ ]:
# load the latest decoder
decoders = glob.glob('decoder_*.pth')
# sort by last modification time
decoders.sort(key=os.path.getmtime)
decoder.load_state_dict(torch.load(decoders[-1]))
print(decoders[-1])
decoder.eval()


decoder_20.pth


/tmp/ipykernel_2257448/3687769428.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoder.load_state_dict(torch.load(decoders[-1]))


Decoder(
  (attn): AttentionFusion(
    (speaker_proj): Linear(in_features=192, out_features=256, bias=True)
    (prosody_proj): Linear(in_features=1024, out_features=256, bias=True)
    (cross_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
    )
    (ffn): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
  )
  (decoder_rnn): AcousticModel(
    (encoder): Encoder(
      (embedding): Embedding(101, 256)
      (prenet): PreNet(
        (net): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): ReLU()
          (2): Dropout(p=0.5, inplace=False)
          (3): Linear(in_features=256, out_features=256, bias=True)
          (4): ReLU()
          (5): Dropout(p=0.5, inplace=False)
        )
      )
      (convs): Sequential(
        (0): Conv1d(256, 512, kernel_

In [514]:
it = iter(test_dl)

In [521]:
batch = next(it)

In [522]:
hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_discrete", trust_repo=True).cuda()

Using cache found in /home/dcor/niskhizov/cache/hub/bshall_hifigan_main


In [532]:
# with torch.no_grad():
        
#         cont_units = decoder.decoder_rnn.encoder(d_units.cuda())

#         units_attn = decoder.attn(cont_units.cuda(), emo_vecs.cuda(), spk_vecs.cuda())  # (batch_size, time, hidden_dim)
        
   
#         o = decoder.decoder_rnn.decoder.generate(units_attn)
wavs,d_units, units_padded, emo_vecs, spk_vecs, logmels_padded, mels_lengths, units_lengths  = batch

decoder = decoder.eval()

with torch.no_grad():
        

        o = decoder.generate(d_units[0].unsqueeze(0).cuda(), emo_vecs[0].unsqueeze(0).cuda(),spk_vecs[0].unsqueeze(0).cuda())
        

In [533]:
spk_vecs[12].shape

torch.Size([192])

In [534]:
# acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_discrete", trust_repo=True).cuda()

# with torch.no_grad():
#     o = acoustic.generate(d_units.cuda())

In [535]:
import plotly.express as px
px.imshow(o[0].detach().cpu().numpy().T)

In [536]:
idx = 0
with torch.no_grad():
    target = hifigan(o[idx,:,:].unsqueeze(0).transpose(1, 2))

In [537]:
Audio(target[0].detach().cpu().numpy(), rate=16000)

In [538]:
Audio(wavs[0],rate=16000)

In [531]:
Audio(wavs[12],rate=16000)

In [379]:
hubert_discrete = torch.hub.load("bshall/hubert:main", "hubert_discrete", trust_repo=True).cuda()


Using cache found in /home/dcor/niskhizov/cache/hub/bshall_hubert_main


In [ ]:
from funasr import AutoModel


In [ ]:
model_id = "iic/emotion2vec_plus_large"

sed_model = AutoModel(
    model=model_id,
    hub="ms",  # "ms" or "modelscope" for China mainland users; "hf" or "huggingface" for other overseas users
)

2025-02-28 12:44:30,720 - modelscope - WARNING - Using branch: master as version is unstable, use with caution


Detect model requirements, begin to install it: /home/dcor/niskhizov/.cache/modelscope/hub/models/iic/emotion2vec_plus_large/requirements.txt
install model requirements successfully
ckpt: /home/dcor/niskhizov/.cache/modelscope/hub/models/iic/emotion2vec_plus_large/model.pt


/home/dcor/niskhizov/anaconda3/lib/python3.12/site-packages/funasr/train_utils/load_pretrained_model.py:68: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



init param, map: modality_encoders.AUDIO.extra_tokens from d2v_model.modality_encoders.AUDIO.extra_tokens in ckpt
init param, map: modality_encoders.AUDIO.alibi_scale from d2v_model.modality_encoders.AUDIO.alibi_scale in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.0.0.weight from d2v_model.modality_encoders.AUDIO.local_encoder.conv_layers.0.0.weight in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.0.2.1.weight from d2v_model.modality_encoders.AUDIO.local_encoder.conv_layers.0.2.1.weight in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.0.2.1.bias from d2v_model.modality_encoders.AUDIO.local_encoder.conv_layers.0.2.1.bias in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.1.0.weight from d2v_model.modality_encoders.AUDIO.local_encoder.conv_layers.1.0.weight in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.1.2.1.weight from d2v_model.modality_encoders.AUDIO.loc

In [ ]:
def extract_embedding(wav_path):
    wav, sr = torchaudio.load(wav_path)

    # take 3 seconds of audio

    with torch.inference_mode():
        # Extract speech units
        discrite_units = hubert_discrete.units(wav.unsqueeze(0).cuda())
        
        emo_vec = torch.tensor(sed_model.generate(wav, granularity="utterance", extract_embedding=True, disable_pbar =True)[0]['feats'])

    return discrite_units, emo_vec, wav



In [ ]:
neutral_wavs = glob.glob('Emotion Speech Dataset/0018/Neutral/*.wav')

In [ ]:
angry_wavs = glob.glob('Emotion Speech Dataset/0018/Angry/*.wav')

In [ ]:
happy_wavs = glob.glob('Emotion Speech Dataset/0018/Happy/*.wav')

In [ ]:
wav_a = "/home/dcor/niskhizov/Prosody2Vec/Emotion Speech Dataset/0018/Sad/0018_001305.wav"
wav_b = angry_wavs[0]
wav_c = happy_wavs[0]
embed_a = extract_embedding(wav_a)
embed_b = extract_embedding(wav_b)
embed_c = extract_embedding(wav_c)


In [ ]:

decoder = decoder.eval()

with torch.no_grad():
        

        o = decoder.generate(embed_a[0].unsqueeze(0).cuda(), embed_b[1].unsqueeze(0).cuda())
        

In [ ]:
with torch.no_grad():
    target = hifigan(o.transpose(1, 2)).cpu()[0][0]

In [ ]:
Audio(target,rate = 16000)

In [ ]:
Audio(embed_a[-1],rate = 16000)

In [ ]:
Audio(embed_b[-1],rate = 16000)

In [ ]:
with torch.no_grad():
        

        o = decoder.generate(embed_a[0].unsqueeze(0).cuda(), embed_c[1].unsqueeze(0).cuda())
        

In [ ]:
with torch.no_grad():
    target = hifigan(o.transpose(1, 2)).cpu()[0][0]

In [ ]:
Audio(target,rate = 16000)


In [ ]:
Audio(embed_a[-1],rate = 16000)

In [ ]:
Audio(embed_b[-1],rate = 16000)